# Drive -> SRT (ruso) con `faster-whisper-xxl` (large-v2)

Notebook autocontenido. Levanta videos desde tu Drive (carpeta **Host Videos**), los transcribe a SRT en ruso con **Faster-Whisper-XXL r245.4** (large-v2 + extraccion de voz Kim2) y guarda los `.srt` en **Subs_RU**.

**Antes de correr:** `Runtime -> Change runtime type -> T4 GPU`.

## Dos modos

- **Celda 1 - automatico (cola por Google Sheet).** Las cuentas se reparten el trabajo solas con un Sheet compartido como cola, coherente entre cuentas al instante (a diferencia de Drive). Autoriza los 2 popups al inicio, baja el binario, y se pone a tomar filas. Recomendado para los 500+.
- **Celdas 2 + 3 - manual por rango en bloque** (fallback).

Los dos modos comparten el mismo motor de transcripcion, con **fallback automatico para videos largos**: si Kim2 muere por OOM en un video largo, el codigo fragmenta el video en silencios (partes <=60 min), transcribe cada parte por separado, y une los SRT con offset al final - todo sin gastar reintentos del Sheet.

## Parametros

```
-m large-v2                 -l ru                 --task transcribe
--initial_prompt None       --reprompt False      --condition_on_previous_text False
--hallucination_silence_threshold 4
--compute_type int8_float16 --temperature 0       --beam_size 5
--vad_filter True
--ff_vocal_extract mdx_kim2 --voc_device cuda     --ff_loudnorm
-f srt                      --max_line_width 200  --max_line_count 1   --sentence
```

> Kim2 (separacion de voz) corre en la misma GPU antes de transcribir; baja un modelo la primera vez y sube el tiempo por video ~1.5x-2x.


## 1) Modo automatico - cola por Google Sheet (RECOMENDADO)

Corre **solo esta celda** en cada cuenta. No hay que editar nada entre cuentas.

**Orden:** autoriza Drive + Sheets (los 2 popups salen **al inicio**, antes de la descarga) -> baja el binario -> entra al loop.

**El Sheet** (`Subs_RU/Status_Videos`) tiene 5 columnas: `video | status | worker | claim_time | tries`. Cada cuenta:

1. Lee toda la tabla (1 request) y toma la primera fila reclamable cuyo video tenga subido: `TODO`, `DOING` vencido (>30 min = huerfano), o `FAILED` con menos de 3 intentos.
2. Marca `DOING + worker + claim_time`, espera 3s, relee: si quedo su worker, gano; si no, va a la siguiente.
3. Transcribe, deja el SRT en `Subs_RU`, marca `DONE`. Si ya existia el SRT, marca `DONE` sin reprocesar.
4. **Fallback OOM:** si Kim2 muere por falta de RAM en un video largo (exit -9/137 + `Kim_Vocal`/`MDX` en stderr), el video se fragmenta en silencios (partes <=60 min con `ffmpeg silencedetect`), cada parte se transcribe por separado y los SRT se unen con offset. **No cuenta como reintento** del Sheet: es parte del mismo intento.
5. **Reintentos:** si falla por algo distinto a OOM, suma 1 a `tries` y marca `FAILED`. Otra cuenta lo retoma hasta llegar a 3 intentos; ahi queda `FAILED` definitivo.
6. **Sin trabajo:** pita y espera 60s. A los **10 chequeos vacios seguidos desconecta el runtime** (`runtime.unassign()`) para no gastar cuota.

> **Setup del Sheet (una vez):** encabezados en la fila 1 (`video | status | worker | claim_time | tries`), los 758 nombres en `A2:A759`, columnas `B..E` vacias. Sheet compartido como editor con las 8 cuentas.


In [ ]:
import os, time, re, subprocess, shutil, random, math
from pathlib import Path
from datetime import datetime, timezone

# ===== Autorizacion: Drive primero (popup al inicio) =====
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
print("Drive montado.")

# ===== Autorizacion: Sheets (popup al inicio, antes de la descarga) =====
from google.colab import auth
auth.authenticate_user()
get_ipython().system("pip install -q --upgrade gspread")
import gspread
from gspread import Cell
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
SHEET_ID  = "10G9lkcjEY_Ns63zWXDTWIo_xT3th044QSYIdyOAqf0M"
SHEET_TAB = "Hoja 1"
ws = gc.open_by_key(SHEET_ID).worksheet(SHEET_TAB)
print("Sheet conectado.")

# ===== Instalacion del binario (descarga larga; ya autorizaste, podes irte) =====
get_ipython().system("apt-get -qq install -y ffmpeg p7zip-full > /dev/null")
import torch
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"
if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, "Descarga incompleta"
    print("Extrayendo (1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), "Extraccion fallo"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[WARN] sin GPU: float16 y la extraccion de voz en cuda van a fallar. Activa T4 GPU.")

# ===== Carpetas y listado =====
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")
assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}

def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.name)]

inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS), key=natural_key)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"
total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}.srt").exists())
print(f"\n{total} archivo(s) en 'Host Videos'  |  {already} con SRT  |  {total-already} pendientes")

TMP_OUT = Path("/content/srt_tmp"); TMP_OUT.mkdir(exist_ok=True)

# ===== Comando XXL + runner =====
def build_cmd(vid, output_dir):
    return [
        str(EXE), str(vid),
        "--model", "large-v2", "--language", "ru", "--task", "transcribe",
        "--initial_prompt", "None", "--reprompt", "False",
        "--condition_on_previous_text", "False",
        "--hallucination_silence_threshold", "4",
        "--compute_type", "int8_float16", "--temperature", "0", "--beam_size", "5",
        "--vad_filter", "True",
        "--ff_vocal_extract", "mdx_kim2", "--voc_device", "cuda", "--ff_loudnorm",
        # "--diarize", "pyannote_v3.1", "--diarize_device", "cuda",
        "--max_line_width", "200", "--max_line_count", "1", "--sentence",
        "--output_dir", str(output_dir), "--output_format", "srt",
    ]

def _run_xxl(vid, output_dir):
    """Corre el CLI. Devuelve (returncode, srt_path|None, stderr).
    Borra SOLO un SRT viejo del mismo stem; NUNCA toca el video de entrada
    (importante cuando 'output_dir' es el mismo dir que contiene las partes)."""
    output_dir.mkdir(parents=True, exist_ok=True)
    srt = output_dir / f"{vid.stem}.srt"
    if srt.exists():
        try: srt.unlink()
        except OSError: pass
    r = subprocess.run(build_cmd(vid, output_dir), capture_output=True, text=True)
    return r.returncode, (srt if srt.exists() else None), r.stderr

# ===== Corte en silencios (fallback si Kim2 OOMs) =====
MAX_PART_SECONDS = 60 * 60   # ajustable; si una parte de 60 min revienta, bajar a 40*60

def _get_video_duration(path):
    r = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", str(path)], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except (ValueError, AttributeError): return 0.0

def _detect_silences(path, noise_db=-30, min_dur=0.5):
    r = subprocess.run(
        ["ffmpeg", "-i", str(path),
         "-af", f"silencedetect=noise={noise_db}dB:d={min_dur}",
         "-f", "null", "-"], capture_output=True, text=True)
    starts = re.findall(r"silence_start:\s*([\d.]+)", r.stderr)
    ends = re.findall(r"silence_end:\s*([\d.]+)", r.stderr)
    return [(float(s), float(e)) for s, e in zip(starts, ends)]

def _plan_cuts(video_path, max_part_sec):
    dur = _get_video_duration(video_path)
    if dur <= max_part_sec: return [], dur
    silences = _detect_silences(video_path)
    n_parts = math.ceil(dur / max_part_sec)
    cuts, prev = [], 0.0
    for i in range(1, n_parts):
        remaining = n_parts - i
        min_cut = max(prev + 1.0, dur - remaining * max_part_sec)
        max_cut = prev + max_part_sec
        best, best_dur = None, -1
        for s, e in silences:
            mid = (s + e) / 2
            if min_cut <= mid <= max_cut and (e - s) > best_dur:
                best_dur, best = e - s, mid
        if best is None:
            best = (min_cut + max_cut) / 2
            print(f"   [WARN] sin silencios entre {min_cut:.0f}s y {max_cut:.0f}s; corte duro a {best:.0f}s")
        cuts.append(round(best, 3))
        prev = best
    return cuts, dur

def _split_video_at(video_path, cuts, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    pattern = str(out_dir / f"{video_path.stem}_part_%03d{video_path.suffix}")
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
           "-i", str(video_path), "-map", "0", "-c", "copy", "-f", "segment",
           "-segment_times", ",".join(f"{c:.3f}" for c in cuts),
           "-segment_start_number", "1", "-reset_timestamps", "1",
           "-avoid_negative_ts", "make_zero", pattern]
    subprocess.run(cmd, check=True)
    parts = sorted(out_dir.glob(f"{video_path.stem}_part_*{video_path.suffix}"))
    if len(parts) != len(cuts) + 1:
        raise RuntimeError(f"Esperaba {len(cuts)+1} partes, salieron {len(parts)}")
    return parts

_srt_lib = None
def _ensure_srt_lib():
    global _srt_lib
    if _srt_lib is None:
        get_ipython().system("pip install -q srt")
        import srt as _s
        _srt_lib = _s
    return _srt_lib

def _merge_srts(part_srts, part_videos, out_path):
    """Une SRTs sumando el offset acumulado (duracion real de cada parte)."""
    srt = _ensure_srt_lib()
    from datetime import timedelta
    all_subs, offset, idx = [], 0.0, 1
    for psrt, pvid in zip(part_srts, part_videos):
        text = psrt.read_text(encoding="utf-8")
        for sub in srt.parse(text):
            all_subs.append(srt.Subtitle(
                index=idx,
                start=sub.start + timedelta(seconds=offset),
                end=sub.end + timedelta(seconds=offset),
                content=sub.content))
            idx += 1
        offset += _get_video_duration(pvid)
    out_path.write_text(srt.compose(all_subs), encoding="utf-8")

def _was_kim2_oom(rc, stderr):
    if rc not in (-9, 137): return False
    s = stderr or ""
    return "Kim_Vocal" in s or "MDX" in s

# ===== Transcribir uno (con fallback split-on-Kim2-OOM) =====
def transcribe_one(vid, label):
    """Transcribe vid; si Kim2 muere por OOM, fragmenta en silencios (<=60 min)
    y reintenta por partes, uniendo los SRT al final."""
    final_srt = OUTPUT_DIR / f"{vid.stem}.srt"
    t0 = time.time()
    rc, srt, stderr = _run_xxl(vid, TMP_OUT)
    if rc == 0 and srt:
        if final_srt.exists():
            srt.unlink(missing_ok=True)
            return "skipped"
        shutil.move(str(srt), str(final_srt))
        n = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
        print(f"   ok {label}: {n} cues, {time.time()-t0:.1f}s -> {final_srt.name}")
        return "done"

    if not _was_kim2_oom(rc, stderr):
        print(f"   x {label}: CLI exit {rc}; no fue OOM en Kim2, no fragmento.")
        if stderr: print(f"   stderr:\n{stderr[-400:]}")
        return "failed"

    print(f"   {label}: OOM en Kim2 -> fragmentando en partes de <={MAX_PART_SECONDS//60} min...")
    try:
        cuts, dur = _plan_cuts(vid, MAX_PART_SECONDS)
    except Exception as ex:
        print(f"   x {label}: error planeando cortes: {ex}"); return "failed"
    if not cuts:
        print(f"   x {label}: video de {dur:.0f}s no requeria split. OOM raro; marco FAILED.")
        return "failed"
    print(f"   Video: {dur/60:.1f} min. {len(cuts)} corte(s) -> {len(cuts)+1} parte(s).")

    split_dir = TMP_OUT / f"{vid.stem}_parts"
    if split_dir.exists(): shutil.rmtree(split_dir, ignore_errors=True)
    try:
        parts = _split_video_at(vid, cuts, split_dir)
    except Exception as ex:
        print(f"   x {label}: error cortando video: {ex}"); return "failed"

    part_srts = []
    for i, part_path in enumerate(parts, 1):
        pdur = _get_video_duration(part_path)
        print(f"   [parte {i}/{len(parts)}] ({pdur/60:.1f} min) {part_path.name}")
        prc, psrt, perr = _run_xxl(part_path, split_dir)
        if prc != 0 or not psrt:
            print(f"   x parte {i}/{len(parts)} fallo (exit {prc}). Abortando split.")
            if perr: print(f"   stderr:\n{perr[-400:]}")
            shutil.rmtree(split_dir, ignore_errors=True)
            return "failed"
        part_srts.append(psrt)

    try:
        _merge_srts(part_srts, parts, final_srt)
    except Exception as ex:
        print(f"   x {label}: error uniendo SRTs: {ex}")
        shutil.rmtree(split_dir, ignore_errors=True)
        return "failed"

    n_cues = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
    print(f"   ok {label}: {n_cues} cues unidas desde {len(parts)} parte(s), {time.time()-t0:.1f}s -> {final_srt.name}")
    shutil.rmtree(split_dir, ignore_errors=True)
    return "done"

# ===== Cola coordinada por Google Sheet =====
import numpy as np
from IPython.display import Audio, display

WORKER    = ""
TTL_MIN   = 30
VERIFY_S  = 3
IDLE_S    = 60
MAX_IDLE  = 10
MAX_TRIES = 3

if not WORKER:
    import uuid; WORKER = "colab-" + uuid.uuid4().hex[:6]
print(f"Worker: {WORKER}")

COL_STATUS, COL_WORKER, COL_TIME, COL_TRIES = 2, 3, 4, 5
local_stems = {p.stem for p in inputs}

def _now(): return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
def _stem(name):
    name = name.strip()
    for e in VIDEO_EXTS:
        if name.lower().endswith(e): return name[:-len(e)]
    return name
def _orphan(ctime):
    try:
        age = (datetime.now(timezone.utc) -
               datetime.strptime(ctime, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)).total_seconds() / 60
        return age > TTL_MIN
    except Exception: return True
def _tries(row): return int(row[4]) if len(row) > 4 and row[4].strip().isdigit() else 0
def beep(freq=880, dur=0.12):
    sr = 8000; t = np.linspace(0, dur, int(sr*dur), endpoint=False)
    display(Audio((0.2*np.sin(2*np.pi*freq*t)).astype(np.float32), rate=sr, autoplay=True))

done = skipped = failed = 0
idle = 0
t_global = time.time()

while True:
    rows = ws.get_all_values()[1:]
    target = None
    for i, row in enumerate(rows, start=2):
        video  = row[0] if len(row) > 0 else ""
        status = (row[1] if len(row) > 1 else "").strip().upper()
        ctime  = row[3] if len(row) > 3 else ""
        tries  = _tries(row)
        if not video:                                continue
        if _stem(video) not in local_stems:          continue
        if status == "DONE":                         continue
        if status == "DOING" and not _orphan(ctime): continue
        if status == "FAILED" and tries >= MAX_TRIES: continue
        target = (i, _stem(video), video, tries)
        break

    if target is None:
        pend = 0
        for r in rows:
            if not (r and r[0]): continue
            st = (r[1] if len(r) > 1 else "").strip().upper()
            if st == "DONE": continue
            if st == "FAILED" and _tries(r) >= MAX_TRIES: continue
            pend += 1
        if pend == 0:
            print("\nTodo terminado."); break
        idle += 1
        beep()
        if idle >= MAX_IDLE:
            print(f"\n{MAX_IDLE} chequeos sin trabajo -> desconecto runtime.")
            from google.colab import runtime; runtime.unassign(); break
        print(f"\nNada para mi ahora ({pend} pendientes). Idle {idle}/{MAX_IDLE}, espero {IDLE_S}s...")
        time.sleep(IDLE_S); continue

    idle = 0
    rownum, stem, video, tries = target

    if (OUTPUT_DIR / f"{stem}.srt").exists():
        ws.update_cells([Cell(rownum, COL_STATUS, "DONE")]); skipped += 1; continue

    ws.update_cells([Cell(rownum, COL_STATUS, "DOING"), Cell(rownum, COL_WORKER, WORKER),
                     Cell(rownum, COL_TIME, _now())])
    time.sleep(VERIFY_S)
    check = ws.row_values(rownum)
    if len(check) < 3 or check[2] != WORKER:
        continue

    print(f"\n[fila {rownum}] (intento {tries+1}/{MAX_TRIES}) Procesando: {video}")
    vid_path = next((p for p in inputs if p.stem == stem), None)
    if vid_path is None:
        ws.update_cells([Cell(rownum, COL_STATUS, "TODO"), Cell(rownum, COL_WORKER, ""),
                         Cell(rownum, COL_TIME, "")]); continue
    try:
        outcome = transcribe_one(vid_path, f"fila {rownum}")
    except Exception as ex:
        print(f"   x excepcion: {ex}"); outcome = "failed"

    if outcome in ("done", "skipped"):
        ws.update_cells([Cell(rownum, COL_STATUS, "DONE"), Cell(rownum, COL_WORKER, WORKER),
                         Cell(rownum, COL_TIME, _now())])
        done += outcome == "done"; skipped += outcome == "skipped"
    else:
        ws.update_cells([Cell(rownum, COL_STATUS, "FAILED"), Cell(rownum, COL_WORKER, WORKER),
                         Cell(rownum, COL_TIME, _now()), Cell(rownum, COL_TRIES, str(tries + 1))])
        failed += 1

print(f"\n=== Fin ({WORKER}) ===")
print(f"  transcritos: {done}  |  saltados: {skipped}  |  fallidos: {failed}")
print(f"  tiempo: {(time.time()-t_global)/60:.1f} min")

sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))


## 2) Setup (modo manual por rango)

Monta Drive (popup al inicio), baja el binario, lista `Host Videos`, sugiere un reparto y te da un cuadro para el rango (ej. `1-100`). Despues, celda 3. Mismo motor que la celda 1, incluyendo el **fallback de fragmentacion ante OOM en Kim2**.

> Si usas el modo automatico (celda 1), ignora las celdas 2 y 3.


In [ ]:
import os, time, re, subprocess, shutil, random, math
from pathlib import Path
from datetime import datetime, timezone

# ===== Autorizacion: Drive primero (popup al inicio) =====
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
print("Drive montado.")

# ===== Instalacion del binario (descarga larga; ya autorizaste, podes irte) =====
get_ipython().system("apt-get -qq install -y ffmpeg p7zip-full > /dev/null")
import torch
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"
if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, "Descarga incompleta"
    print("Extrayendo (1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), "Extraccion fallo"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[WARN] sin GPU: float16 y la extraccion de voz en cuda van a fallar. Activa T4 GPU.")

# ===== Carpetas y listado =====
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")
assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}

def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.name)]

inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS), key=natural_key)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"
total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}.srt").exists())
print(f"\n{total} archivo(s) en 'Host Videos'  |  {already} con SRT  |  {total-already} pendientes")

TMP_OUT = Path("/content/srt_tmp"); TMP_OUT.mkdir(exist_ok=True)

# ===== Comando XXL + runner =====
def build_cmd(vid, output_dir):
    return [
        str(EXE), str(vid),
        "--model", "large-v2", "--language", "ru", "--task", "transcribe",
        "--initial_prompt", "None", "--reprompt", "False",
        "--condition_on_previous_text", "False",
        "--hallucination_silence_threshold", "4",
        "--compute_type", "int8_float16", "--temperature", "0", "--beam_size", "5",
        "--vad_filter", "True",
        "--ff_vocal_extract", "mdx_kim2", "--voc_device", "cuda", "--ff_loudnorm",
        # "--diarize", "pyannote_v3.1", "--diarize_device", "cuda",
        "--max_line_width", "200", "--max_line_count", "1", "--sentence",
        "--output_dir", str(output_dir), "--output_format", "srt",
    ]

def _run_xxl(vid, output_dir):
    """Corre el CLI. Devuelve (returncode, srt_path|None, stderr).
    Borra SOLO un SRT viejo del mismo stem; NUNCA toca el video de entrada
    (importante cuando 'output_dir' es el mismo dir que contiene las partes)."""
    output_dir.mkdir(parents=True, exist_ok=True)
    srt = output_dir / f"{vid.stem}.srt"
    if srt.exists():
        try: srt.unlink()
        except OSError: pass
    r = subprocess.run(build_cmd(vid, output_dir), capture_output=True, text=True)
    return r.returncode, (srt if srt.exists() else None), r.stderr

# ===== Corte en silencios (fallback si Kim2 OOMs) =====
MAX_PART_SECONDS = 60 * 60   # ajustable; si una parte de 60 min revienta, bajar a 40*60

def _get_video_duration(path):
    r = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", str(path)], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except (ValueError, AttributeError): return 0.0

def _detect_silences(path, noise_db=-30, min_dur=0.5):
    r = subprocess.run(
        ["ffmpeg", "-i", str(path),
         "-af", f"silencedetect=noise={noise_db}dB:d={min_dur}",
         "-f", "null", "-"], capture_output=True, text=True)
    starts = re.findall(r"silence_start:\s*([\d.]+)", r.stderr)
    ends = re.findall(r"silence_end:\s*([\d.]+)", r.stderr)
    return [(float(s), float(e)) for s, e in zip(starts, ends)]

def _plan_cuts(video_path, max_part_sec):
    dur = _get_video_duration(video_path)
    if dur <= max_part_sec: return [], dur
    silences = _detect_silences(video_path)
    n_parts = math.ceil(dur / max_part_sec)
    cuts, prev = [], 0.0
    for i in range(1, n_parts):
        remaining = n_parts - i
        min_cut = max(prev + 1.0, dur - remaining * max_part_sec)
        max_cut = prev + max_part_sec
        best, best_dur = None, -1
        for s, e in silences:
            mid = (s + e) / 2
            if min_cut <= mid <= max_cut and (e - s) > best_dur:
                best_dur, best = e - s, mid
        if best is None:
            best = (min_cut + max_cut) / 2
            print(f"   [WARN] sin silencios entre {min_cut:.0f}s y {max_cut:.0f}s; corte duro a {best:.0f}s")
        cuts.append(round(best, 3))
        prev = best
    return cuts, dur

def _split_video_at(video_path, cuts, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    pattern = str(out_dir / f"{video_path.stem}_part_%03d{video_path.suffix}")
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
           "-i", str(video_path), "-map", "0", "-c", "copy", "-f", "segment",
           "-segment_times", ",".join(f"{c:.3f}" for c in cuts),
           "-segment_start_number", "1", "-reset_timestamps", "1",
           "-avoid_negative_ts", "make_zero", pattern]
    subprocess.run(cmd, check=True)
    parts = sorted(out_dir.glob(f"{video_path.stem}_part_*{video_path.suffix}"))
    if len(parts) != len(cuts) + 1:
        raise RuntimeError(f"Esperaba {len(cuts)+1} partes, salieron {len(parts)}")
    return parts

_srt_lib = None
def _ensure_srt_lib():
    global _srt_lib
    if _srt_lib is None:
        get_ipython().system("pip install -q srt")
        import srt as _s
        _srt_lib = _s
    return _srt_lib

def _merge_srts(part_srts, part_videos, out_path):
    """Une SRTs sumando el offset acumulado (duracion real de cada parte)."""
    srt = _ensure_srt_lib()
    from datetime import timedelta
    all_subs, offset, idx = [], 0.0, 1
    for psrt, pvid in zip(part_srts, part_videos):
        text = psrt.read_text(encoding="utf-8")
        for sub in srt.parse(text):
            all_subs.append(srt.Subtitle(
                index=idx,
                start=sub.start + timedelta(seconds=offset),
                end=sub.end + timedelta(seconds=offset),
                content=sub.content))
            idx += 1
        offset += _get_video_duration(pvid)
    out_path.write_text(srt.compose(all_subs), encoding="utf-8")

def _was_kim2_oom(rc, stderr):
    if rc not in (-9, 137): return False
    s = stderr or ""
    return "Kim_Vocal" in s or "MDX" in s

# ===== Transcribir uno (con fallback split-on-Kim2-OOM) =====
def transcribe_one(vid, label):
    """Transcribe vid; si Kim2 muere por OOM, fragmenta en silencios (<=60 min)
    y reintenta por partes, uniendo los SRT al final."""
    final_srt = OUTPUT_DIR / f"{vid.stem}.srt"
    t0 = time.time()
    rc, srt, stderr = _run_xxl(vid, TMP_OUT)
    if rc == 0 and srt:
        if final_srt.exists():
            srt.unlink(missing_ok=True)
            return "skipped"
        shutil.move(str(srt), str(final_srt))
        n = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
        print(f"   ok {label}: {n} cues, {time.time()-t0:.1f}s -> {final_srt.name}")
        return "done"

    if not _was_kim2_oom(rc, stderr):
        print(f"   x {label}: CLI exit {rc}; no fue OOM en Kim2, no fragmento.")
        if stderr: print(f"   stderr:\n{stderr[-400:]}")
        return "failed"

    print(f"   {label}: OOM en Kim2 -> fragmentando en partes de <={MAX_PART_SECONDS//60} min...")
    try:
        cuts, dur = _plan_cuts(vid, MAX_PART_SECONDS)
    except Exception as ex:
        print(f"   x {label}: error planeando cortes: {ex}"); return "failed"
    if not cuts:
        print(f"   x {label}: video de {dur:.0f}s no requeria split. OOM raro; marco FAILED.")
        return "failed"
    print(f"   Video: {dur/60:.1f} min. {len(cuts)} corte(s) -> {len(cuts)+1} parte(s).")

    split_dir = TMP_OUT / f"{vid.stem}_parts"
    if split_dir.exists(): shutil.rmtree(split_dir, ignore_errors=True)
    try:
        parts = _split_video_at(vid, cuts, split_dir)
    except Exception as ex:
        print(f"   x {label}: error cortando video: {ex}"); return "failed"

    part_srts = []
    for i, part_path in enumerate(parts, 1):
        pdur = _get_video_duration(part_path)
        print(f"   [parte {i}/{len(parts)}] ({pdur/60:.1f} min) {part_path.name}")
        prc, psrt, perr = _run_xxl(part_path, split_dir)
        if prc != 0 or not psrt:
            print(f"   x parte {i}/{len(parts)} fallo (exit {prc}). Abortando split.")
            if perr: print(f"   stderr:\n{perr[-400:]}")
            shutil.rmtree(split_dir, ignore_errors=True)
            return "failed"
        part_srts.append(psrt)

    try:
        _merge_srts(part_srts, parts, final_srt)
    except Exception as ex:
        print(f"   x {label}: error uniendo SRTs: {ex}")
        shutil.rmtree(split_dir, ignore_errors=True)
        return "failed"

    n_cues = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
    print(f"   ok {label}: {n_cues} cues unidas desde {len(parts)} parte(s), {time.time()-t0:.1f}s -> {final_srt.name}")
    shutil.rmtree(split_dir, ignore_errors=True)
    return "done"

# ===== Reparto sugerido + cuadro de rango (modo manual) =====
import ipywidgets as widgets
from IPython.display import display
N_ACCOUNTS = 5
chunk = (total + N_ACCOUNTS - 1) // N_ACCOUNTS
print(f"\nReparto sugerido para {N_ACCOUNTS} cuentas (~{chunk} c/u):")
for k in range(N_ACCOUNTS):
    a = k * chunk + 1; b = min((k + 1) * chunk, total)
    if a > total: break
    print(f"   cuenta {k+1} -> {a}-{b}")
range_box = widgets.Text(value=f"1-{min(100, total)}", placeholder="ej: 1-100 (o 'all')",
                         description="Rango:", layout=widgets.Layout(width="60%"),
                         style={"description_width": "60px"})
print(f"\nElegi el rango (1..{total}) y pasa a la celda 3:")
display(range_box)


## 3) Transcribir un rango (modo manual)

Lee el rango del cuadro de la celda 2 y transcribe solo esos. Salta los que ya tienen SRT.


In [ ]:
import numpy as np
from IPython.display import Audio, display
assert "range_box" in globals(), "Primero corre la celda 2 (setup manual)."
spec = (range_box.value or "").strip().lower()
if spec in ("", "all"):
    start, end = 1, total
else:
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", spec) or re.fullmatch(r"(\d+)", spec)
    assert m, f"Rango invalido: {spec!r}. Usa '1-100' o '50' o 'all'."
    start, end = (int(m.group(1)), int(m.group(1))) if m.lastindex == 1 else (int(m.group(1)), int(m.group(2)))
start = max(1, start); end = min(total, end)
assert start <= end, f"Rango vacio tras recortar a 1..{total}: {start}-{end}"
batch = inputs[start-1:end]
print(f"Procesando {len(batch)} archivo(s): #{start} a #{end} de {total}.")
done = skipped = failed = 0; t_global = time.time()
for offset, vid in enumerate(batch):
    i = start + offset
    if (OUTPUT_DIR / f"{vid.stem}.srt").exists():
        print(f"\n[{i}/{end}] SALTADO (ya existe): {vid.stem}.srt"); skipped += 1; continue
    print(f"\n[{i}/{end}] Procesando: {vid.name}")
    try: outcome = transcribe_one(vid, f"{i}/{end}")
    except Exception as ex: print(f"   x excepcion: {ex}"); outcome = "failed"
    done += outcome == "done"; skipped += outcome == "skipped"; failed += outcome == "failed"
print(f"\n=== Rango {start}-{end}: {done} ok, {skipped} saltados, {failed} fallidos, {(time.time()-t_global)/60:.1f} min ===")
sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))
